# GPU Utilization Report

This notebook demonstrates how to monitor and report GPU utilization during deep learning training.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import time
import pandas as pd

# Check if CUDA is available
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not available.")

device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

# Define transforms
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
])

# Load dataset
dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

# Create data loader
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

# Load model
model = torchvision.models.resnet50().to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# List to store utilization metrics
metrics = []

# Training loop
model.train()
for epoch in range(2):  # Just a couple of epochs for demonstration
    for batch_idx, (data, target) in enumerate(loader):
        if batch_idx >= 10:  # Limit batches for quick report
            break
        data, target = data.to(device), target.to(device)
        
        # Record start time and memory
        start_time = time.time()
        torch.cuda.synchronize()
        mem_start = torch.cuda.memory_allocated()
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        torch.cuda.synchronize()
        end_time = time.time()
        mem_end = torch.cuda.memory_allocated()
        
        # Calculate metrics
        batch_time = end_time - start_time
        mem_used = mem_end - mem_start
        utilization = (mem_used / torch.cuda.get_device_properties(0).total_memory) * 100
        
        metrics.append({
            'epoch': epoch,
            'batch': batch_idx,
            'batch_time_sec': batch_time,
            'memory_used_mb': mem_used / (1024 * 1024),
            'memory_utilization_percent': utilization
        })
        
        if batch_idx % 5 == 0:
            print(f'Epoch {epoch}, Batch {batch_idx}, Time: {batch_time:.4f}s, Mem: {mem_used/(1024*1024):.2f} MB')

# Convert to DataFrame and display
df = pd.DataFrame(metrics)
print(df.head())

# Save to CSV
os.makedirs('results/profiler_reports', exist_ok=True)
df.to_csv('results/profiler_reports/gpu_utilization.csv', index=False)
print("\nSaved GPU utilization report to: results/profiler_reports/gpu_utilization.csv")